# Notebook 02 — GraphRAG Retriever Development

Builds sentence-transformer embeddings, validates seed retrieval, and compares Flat RAG vs GraphRAG subgraph quality using GGS.

**No GPU required** (MiniLM embeddings run fine on CPU for the IFC graph size).

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os, sys
REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'   # paths after chdir must not include repo name
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

import networkx as nx
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from evaluation.metrics.ggs import GGSScorer
from benchmark.dtah_bench import DTAHBench

IFC_PATH    = 'benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = 'outputs/graphs/ifc_graph.json'
EMB_CACHE   = 'outputs/embedders/graph_embedder'
os.makedirs('outputs/embedders', exist_ok=True)
os.makedirs('outputs/figures',   exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Load IFC graph (build if not cached) ──────────────────────────────
if not Path(IFC_PATH).exists():
    import urllib.request
    print('Downloading duplex.ifc...')
    urllib.request.urlretrieve(
        'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc',
        IFC_PATH)

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
    print(f'Graph loaded from cache: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
else:
    print('Building graph (run Notebook 01 first to cache it)...')
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)
    print(f'Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# ── Cell 3: Build or load graph embedder ─────────────────────────────────────
if Path(EMB_CACHE).exists() and Path(f'{EMB_CACHE}/faiss.index').exists():
    print('Loading cached embedder...')
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embeddings — sentence-transformers/all-MiniLM-L6-v2 on CPU...')
    print('This takes 2–5 minutes. Will be cached for future runs.')
    embedder = GraphEmbedder(model_name='sentence-transformers/all-MiniLM-L6-v2')
    embedder.fit(G)
    embedder.save(EMB_CACHE)
    print(f'Embedder saved: {EMB_CACHE}')

print(f'Embedder ready: {len(embedder._node_ids)} nodes indexed')

In [ ]:
# ── Cell 4: Seed retrieval quality test ──────────────────────────────────────
TEST_PROMPTS = [
    ('T1-MEP-001', 'Generate a centrifugal pump',                                             ['IfcPump','IfcFlowMovingDevice']),
    ('T1-MEP-002', 'Generate a steel globe valve',                                            ['IfcValve']),
    ('T2-MEP-001', 'Generate a centrifugal pump connected to two pipe segments',              ['IfcPump','IfcPipeSegment']),
    ('T2-HVAC-001','Generate an AHU connected to a supply air duct',                          ['IfcAirToAirHeatRecovery','IfcDuctSegment']),
    ('T3-MEP-001', 'Generate a pump room with two parallel pumps and isolation valves',       ['IfcPump','IfcValve']),
]

print(f'{"":<2} {"Prompt ID":<14} {"Top-5 retrieved types"}')
print('-' * 80)
for pid, prompt, expected in TEST_PROMPTS:
    seeds = embedder.retrieve_seeds(prompt, top_k=5)
    retrieved_types = [s['ifc_type'] for s in seeds]
    hit = any(e in retrieved_types for e in expected)
    flag = '✓' if hit else '⚠'
    print(f'{flag}  {pid:<14} {retrieved_types}')
    print(f'     Expected any of: {expected}')
    print()

In [ ]:
# ── Cell 5: Flat RAG vs GraphRAG comparison — the key experiment ──────────────
test_prompt = 'Generate a pump room with two pumps in parallel, inlet and outlet headers, and isolation valves'
print(f'Test prompt:\n  {test_prompt}\n')

# Shared seeds
seeds    = embedder.retrieve_seeds(test_prompt, top_k=5)
seed_ids = [s['node_id'] for s in seeds]
print(f'Seeds retrieved: {[s["ifc_type"] for s in seeds]}')

# Flat RAG: induced subgraph on seed nodes only — no traversal
flat_subgraph = G.subgraph(seed_ids).copy()

# GraphRAG: k-hop traversal from the same seeds
traversal    = KHopTraversal(G, max_depth=3, bidirectional=True)
trav_result  = traversal.traverse(seed_ids)
graph_subgraph = trav_result.subgraph

print(f'\n{"Metric":<32} {"Flat RAG":>10} {"GraphRAG":>10}')
print('-' * 56)
print(f'{"Nodes recovered":<32} {flat_subgraph.number_of_nodes():>10} {graph_subgraph.number_of_nodes():>10}')
print(f'{"Edges recovered":<32} {flat_subgraph.number_of_edges():>10} {graph_subgraph.number_of_edges():>10}')

flat_rels  = set(d.get('relation_type','') for _,_,d in flat_subgraph.edges(data=True))
graph_rels = set(d.get('relation_type','') for _,_,d in graph_subgraph.edges(data=True))
print(f'{"Relation types found":<32} {len(flat_rels):>10} {len(graph_rels):>10}')
print(f'\nFlat RAG edges: {flat_rels or "NONE — all edges lost"}')
print(f'GraphRAG edges (sample): {list(graph_rels)[:4]}')
print('\n→ Flat RAG recovers nodes but LOSES all relational structure.')
print('→ GraphRAG recovers both entities and typed IFC relations.')

In [ ]:
# ── Cell 6: GGS on pilot benchmark prompts ────────────────────────────────────
bench = DTAHBench(pilot_mode=True)
ggs_scorer = GGSScorer()
TIER_DEPTHS = {1: 1, 2: 2, 3: 4}

results = []
for tier in [1, 2, 3]:
    prompts = bench.load_tier(tier)[:5]  # 5 per tier for this notebook
    for p in prompts:
        seeds    = embedder.retrieve_seeds(p['prompt'], top_k=5)
        seed_ids = [s['node_id'] for s in seeds]

        flat_sg  = G.subgraph(seed_ids).copy()

        depth    = TIER_DEPTHS[tier]
        trav     = KHopTraversal(G, max_depth=depth, bidirectional=True)
        graph_sg = trav.traverse(seed_ids).subgraph

        # Compare flat vs graph against graph as proxy GT
        ggs_flat  = ggs_scorer.score(flat_sg,  graph_sg)
        ggs_graph = ggs_scorer.score(graph_sg, graph_sg)

        results.append({
            'id': p['id'], 'tier': tier,
            'flat_total': ggs_flat.total,   'flat_edge': ggs_flat.edge_recall,
            'graph_total': ggs_graph.total, 'graph_edge': ggs_graph.edge_recall,
        })

print(f'{"ID":<15} {"T":>2} {"Flat GGS":>10} {"Graph GGS":>10} {"Flat EdgeR":>11} {"Graph EdgeR":>12}')
print('-' * 68)
for r in results:
    print(f'{r["id"]:<15} {r["tier"]:>2} {r["flat_total"]:>10.3f} {r["graph_total"]:>10.3f} '
          f'{r["flat_edge"]:>11.3f} {r["graph_edge"]:>12.3f}')

In [ ]:
# ── Cell 7: GGS by tier bar chart ────────────────────────────────────────────
tiers = [1, 2, 3]
flat_means  = [np.mean([r['flat_total']  for r in results if r['tier']==t]) for t in tiers]
graph_means = [np.mean([r['graph_total'] for r in results if r['tier']==t]) for t in tiers]

x = np.arange(len(tiers))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, flat_means,  w, label='Flat RAG',   color='#888780', edgecolor='white')
ax.bar(x + w/2, graph_means, w, label='GraphRAG-DT', color='#2E5FA3', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(['Tier 1\n(Asset)', 'Tier 2\n(Assembly)', 'Tier 3\n(System)'])
ax.set_ylabel('GGS (Graph Grounding Score)')
ax.set_ylim(0, 1.05)
ax.set_title('Flat RAG vs GraphRAG-DT — GGS by Tier')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/02_ggs_comparison.png', dpi=150)
plt.show()
print('Figure saved: outputs/figures/02_ggs_comparison.png')
print('\nRetriever validation complete. Proceed to Notebook 03.')